# **Random Forest - Classificação**


Vou implementar um modelo de classificação Random Forest utilizando a base de leads de uma loja automobilistica. Vou explorar como o Random Forest pode melhorar a precisão das previsões e a robustez do modelo. Além disso, irei ajustar hiperparâmetros para otimizar o desempenho do modelo. Essa prática reforçará a teoria aprendida e demonstrará a aplicação prática desse algoritmo poderoso.

In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import RandomizedSearchCV
import warnings 

warnings.filterwarnings("ignore")

In [2]:
base = pd.read_csv("../../Base de Dados/CARRO_CLIENTES.csv", delimiter=',')

In [3]:
# Substituindo os valores da coluna 'GENDER' por números específicos utilizando o replace
gender_mapping = {'Male': 0, 'Female': 1}
base['Gender'] = base['Gender'].replace(gender_mapping)

In [4]:
# Separando em X (variáveis de entrada) e Y (variável de saída)
X = base.drop('Purchased', axis=1)  # X contém todas as colunas exceto 'Purchased'
Y = base['Purchased']  # Y contém apenas a coluna 'Purchased'

In [5]:
# Separar em base de treino e teste (usando 80% para treino e 20% para teste)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

Random Forest geralmente se adapta melhor a dados despadronizados em comparação com a regressão logística. Isso ocorre porque o Random Forest é baseado em árvores de decisão, que são menos sensíveis à escala dos dados.

Aplicando nosso modelo sem ajustar hyperparametros:


In [6]:
# Iniciando o modelo Random Forest
rf_model = RandomForestClassifier(random_state=42)

In [7]:
# Treinando o modelo
rf_model.fit(X_train, Y_train)

RandomForestClassifier(random_state=42)

In [8]:
# Fazendo previsões no conjunto de teste
Y_pred = rf_model.predict(X_test)

In [9]:
# Avaliando o modelo
accuracy = accuracy_score(Y_test, Y_pred)
report = classification_report(Y_test, Y_pred)
conf_matrix = confusion_matrix(Y_test, Y_pred)

print(f"Acurácia: {accuracy:.2f}")
print("Relatório de Classificação:\n", report)
print("Matriz de Confusão:\n", conf_matrix)

Acurácia: 0.91
Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.91      0.93      0.92       112
           1       0.91      0.89      0.90        88

    accuracy                           0.91       200
   macro avg       0.91      0.91      0.91       200
weighted avg       0.91      0.91      0.91       200

Matriz de Confusão:
 [[104   8]
 [ 10  78]]


Precisão: Para ambas as classes, 91% das previsões positivas foram corretas.

Recall: Para a classe 0, 93% dos casos reais positivos foram identificados corretamente, enquanto para a classe 1, 89% foram identificados corretamente.

F1: Para a classe 0, o F1-Score é 0.92, e para a classe 1, é 0.90.

Verdadeiros Negativos: 104
Falsos positivos: 8
Falsos Negativos: 10
Verdadeiros Positivos: 78

Como o objetivo é identificar possíveis compradores de carros a partir dos leads de clientes que entram no site, para enviar eventos promocionais.

**Falsos Positivos:** Clientes que são classificados como potenciais compradores (classe 1), mas que na verdade não comprarão um carro (classe 0).

Impacto: Enviar eventos promocionais para esses clientes que não têm intenção de comprar.
- Custo de envio de promoções e recursos de marketing desnecessários.

**Falsos Negativos:** Clientes que são classificados como não compradores (classe 0), mas que na verdade comprariam um carro (classe 1).

Impacto: Perder oportunidades de conversão porque os clientes que poderiam ser compradores não receberam eventos promocionais.
- Perda de receita potencial devido a oportunidades perdidas.


## **Melhorando o Modelo**

**Definir o Espaço de Busca dos Hiperparâmetros:** 

Alguns hiperparâmetros importantes para o Random Forest incluem:

n_estimators: Número de árvores na floresta.

max_depth: Profundidade máxima das árvores.

min_samples_split: Número mínimo de amostras necessárias para dividir um nó.

min_samples_leaf: Número mínimo de amostras necessárias para estar em um nó folha.

max_features: Número de recursos a serem considerados para encontrar a melhor divisão.

In [10]:
# Definir o espaço de busca dos hiperparâmetros
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

In [11]:
# Configurando o Randomized Search
random_search = RandomizedSearchCV(estimator=rf_model, param_distributions=param_grid,
                                   n_iter=100, cv=5, n_jobs=-1, verbose=2, random_state=42, scoring='accuracy')

In [12]:
# Executando o Randomized Search
random_search.fit(X_train, Y_train)

Fitting 5 folds for each of 100 candidates, totalling 500 fits
[CV] END max_depth=10, max_features=log2, min_samples_leaf=4, min_samples_split=10, n_estimators=50; total time=   0.1s
[CV] END max_depth=10, max_features=log2, min_samples_leaf=1, min_samples_split=2, n_estimators=50; total time=   0.1s
[CV] END max_depth=10, max_features=log2, min_samples_leaf=4, min_samples_split=10, n_estimators=50; total time=   0.2s
[CV] END max_depth=10, max_features=log2, min_samples_leaf=1, min_samples_split=2, n_estimators=50; total time=   0.2s
[CV] END max_depth=10, max_features=log2, min_samples_leaf=1, min_samples_split=2, n_estimators=50; total time=   0.1s
[CV] END max_depth=10, max_features=log2, min_samples_leaf=4, min_samples_split=10, n_estimators=50; total time=   0.2s
[CV] END max_depth=10, max_features=log2, min_samples_leaf=4, min_samples_split=10, n_estimators=50; total time=   0.2s
[CV] END max_depth=10, max_features=log2, min_samples_leaf=1, min_samples_split=2, n_estimators=50; 

RandomizedSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42),
                   n_iter=100, n_jobs=-1,
                   param_distributions={'max_depth': [None, 10, 20, 30],
                                        'max_features': ['sqrt', 'log2', None],
                                        'min_samples_leaf': [1, 2, 4],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [50, 100, 200]},
                   random_state=42, scoring='accuracy', verbose=2)

In [13]:
# Obtendo os melhores hiperparâmetros através do método randomico
best_params = random_search.best_params_
print(f"Melhores Hiperparâmetros: {best_params}")

Melhores Hiperparâmetros: {'n_estimators': 50, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 10}


**n_estimators -  50**

Descrição: Este parâmetro define o número de árvores na floresta.


**min_samples_split- 10**

Descrição: Este parâmetro define o número mínimo de amostras necessárias para dividir um nó.

**min_samples_leaf- 1**

Descrição: Este parâmetro especifica o número mínimo de amostras que um nó folha deve ter.


**max_features - 'log2'**

Descrição: Este parâmetro define o número de características a serem consideradas ao procurar a melhor divisão. Com log2, o modelo considera o logaritmo de base 2 do número total de características.

**max_depth: - 10**

Descrição: Este parâmetro especifica a profundidade máxima das árvores na floresta.

In [14]:
# Treinando o modelo com os melhores hiperparâmetros encontrados acima
best_rf_model = random_search.best_estimator_
best_rf_model.fit(X_train, Y_train)

RandomForestClassifier(max_depth=10, max_features='log2', min_samples_split=10,
                       n_estimators=50, random_state=42)

In [15]:
Y_pred = best_rf_model.predict(X_test)

In [16]:
accuracy = accuracy_score(Y_test, Y_pred)
report = classification_report(Y_test, Y_pred)
conf_matrix = confusion_matrix(Y_test, Y_pred)

print(f"Acurácia: {accuracy:.2f}")
print("Relatório de Classificação:\n", report)
print("Matriz de Confusão:\n", conf_matrix)

Acurácia: 0.91
Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.91      0.94      0.92       112
           1       0.92      0.88      0.90        88

    accuracy                           0.91       200
   macro avg       0.91      0.91      0.91       200
weighted avg       0.91      0.91      0.91       200

Matriz de Confusão:
 [[105   7]
 [ 11  77]]
